In [3]:
from behave_analysis.database.Experiments.JAL003_ex import JAL3_7sept, JAL3_4sept

from behave_analysis.database.Experiments.JAL004_ex import JAL4_3rdSept, JAL4_19thSept, JAL4_28aug, JAL4_11thSept

from behave_analysis.database.Experiments.JAL005_ex import JAL005_8thSept, JAL005_21stSept

from behave_analysis.database.Experiments.JAL006_ex import JAL6_28mar, JAL6_flip4_21mar, JAL6_flip5_25mar, JAL6_flip3_18mar, JAL6_flip7_1apr

from behave_analysis.database.Experiments.JAL007_ex import JAL7_sesh8_9apr, JAL7_sesh9_16apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_23apr,JAL7_30apr

from behave_analysis.database.Experiments.JAL008_ex import JAL8_flip1_25apr, JAL8_flip2_29apr, JAL8_tiny_3may, JAL8_flip4_10may, JAL8_14may, JAL8_21may

experiments_objects = [JAL3_7sept, JAL3_4sept, 
JAL4_3rdSept, JAL4_19thSept, JAL4_28aug, JAL4_11thSept,
JAL005_8thSept, JAL005_21stSept, 
JAL6_28mar, JAL6_flip4_21mar, JAL6_flip5_25mar, JAL6_flip3_18mar, JAL6_flip7_1apr,
JAL7_sesh8_9apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_sesh9_16apr, JAL7_23apr, JAL7_30apr,
JAL8_flip1_25apr,JAL8_flip2_29apr, JAL8_tiny_3may, JAL8_flip4_10may, JAL8_14may, JAL8_21may]

In [ ]:
from behave_analysis.process.process import Process
from behave_analysis.visualize.visualize_utils import open_tracking_data
from behave_analysis.analyze.behaviour.utils import base_plotting, 
from behave_analysis.utils.identify_condition import identify_condition_of_trial
from behave_analysis.utils.arena_plotting import Arena

import os
import dill as pickle
import numpy as np
import matplotlib.pyplot as plt
import re
from sklearn.metrics.pairwise import cosine_similarity
import polars as pl

In [37]:
session = Process(experiments_objects[9]).load_session()
onset_frames = [x[0] for x in session.audio.onset_frames]
stimulus_durations = [x[0] for x in session.audio.stimulus_durations]
video_df = pl.read_csv(os.path.join(session.base_path, session.processed_path) + "\\" "full_video_dataframe.csv")
tracking_data = open_tracking_data(session)

In [41]:
onset_frame = onset_frames[4]
stimulus_duration = stimulus_durations[4]
x_loc = tracking_data['head_loc'][onset_frame:onset_frame + int(stimulus_duration*session.video.fps),0]
y_loc = tracking_data['head_loc'][onset_frame:onset_frame + int(stimulus_duration*session.video.fps),1]
in_shelt = np.where(y_loc > tracking_data['shelter_loc'][0][1])[0]
if len(in_shelt)>0:
    xx = x_loc[:in_shelt[0]]
    yy = y_loc[:in_shelt[0]]
plt.plot(x_loc,y_loc)
plt.plot(xx,yy)
plt.show()

In [44]:
ntrial = len(onset_frames)
nrows = 3
ncols = ntrial // nrows + (ntrial % nrows > 0)

for trial_num, (onset_frame, stimulus_duration) in enumerate(zip(onset_frames, stimulus_durations)):
    condition = identify_condition_of_trial(video_df.filter(video_df["frames"] == onset_frame), session)
    # real trajectory
    x_loc = tracking_data['head_loc'][onset_frame:onset_frame + int(stimulus_duration*session.video.fps),0]
    y_loc = tracking_data['head_loc'][onset_frame:onset_frame + int(stimulus_duration*session.video.fps),1]
    # crop the points after the mouse has entered the shelter
    in_shelt = np.where(y_loc > tracking_data['shelter_loc'][0][1])[0]
    if len(in_shelt)>0:
        x_loc = x_loc[:in_shelt[0]]
        y_loc = y_loc[:in_shelt[0]]
    # interpolate to standard size
    x_loc = np.interp(np.arange(0,len(x_loc),len(x_loc)/100),np.arange(len(x_loc)),x_loc)
    y_loc = np.interp(np.arange(0,len(y_loc),len(y_loc)/100),np.arange(len(y_loc)),y_loc)

    # optimal
    opt_x = [tracking_data['head_loc'][onset_frame,0]]
    opt_y = [tracking_data['head_loc'][onset_frame,1]]
    opt_t = [0]
    if not(np.logical_or(condition == 'shelter_only', condition == 'barrier_removed')):
        opt_t = np.append(opt_t,(len(x_loc)-1)/2)
        # mouse_south_of_barrier = np.where(y_loc > 512)[0]
        # if len(mouse_south_of_barrier)>0:
        #     opt_t = np.append(opt_t,mouse_south_of_barrier[0])
        # else:
        #     opt_t = np.append(opt_t,(len(x_loc)-1)/2)
        if condition == 'barrier_pre_flip':
            opt_x = np.append(opt_x,tracking_data['barrier_loc'][0][0])
            opt_y = np.append(opt_y,tracking_data['barrier_loc'][0][1])
        if condition == 'barrier_post_flip':
            opt_x = np.append(opt_x,tracking_data['barrier_loc'][1][0])
            opt_y = np.append(opt_y,tracking_data['barrier_loc'][1][1])

    opt_x = np.append(opt_x,np.mean([tracking_data['shelter_loc'][0][0],tracking_data['shelter_loc'][1][0]]))
    opt_y = np.append(opt_y,tracking_data['shelter_loc'][0][1])
    opt_t = np.append(opt_t,len(x_loc)-1)
    
    # interpolate optimal
    t_int = np.arange(len(x_loc))
    opt_xn = np.interp(t_int,opt_t,opt_x)
    opt_yn = np.interp(t_int,opt_t,opt_y)

    # cosine similarity
    cs = []
    for x,y,ox,oy in zip(x_loc,y_loc,opt_xn,opt_yn):
        cs = np.append(cs,cosine_similarity(np.array([x - 512,y]).reshape(1, -1),np.array([ox - 512,oy]).reshape(1, -1)))

    axs = plt.subplot(nrows, ncols, trial_num + 1)
    Arena(ax=axs, shelter_coordinates=tracking_data["shelter_loc"], condition=condition, barrier_coordinates=session.barrier_location)
    # base_plotting(axs,tracking_data,condition, session = session)
    axs.scatter(x_loc,y_loc,s=3,c = cs)
    axs.scatter(opt_xn,opt_yn, s=3)
    axs.set_xlim([0,1024])
    axs.set_ylim([0,1024])
    axs.invert_yaxis()
    # axs.set_title('x = '+str(cs[0,0])+' , y = '+str(cs[1,1]))
    axs.set_title(np.mean(cs))
plt.show()

In [19]:
x_loc

-512

In [98]:
tracking_data['shelter_loc'][0][1]

886

In [91]:
np.array([x_loc[0],y_loc[0]]).reshape(1, -1)

array([[499.75599583,  95.13672066]])

In [94]:
cosine_similarity(np.array([x_loc[100],y_loc[100]]).reshape(1, -1),np.array([opt_xn[100],opt_yn[100]]).reshape(1, -1))

array([[0.99996907]])